# Project 3: A Financial Event in Five Charts

**DSC 682 Data Visualization for Business Analytics, Fall 2026 · Groups of 3**

Pick one option and tell the story so that a reader who knows nothing arrives at your conclusion. Five charts, each answering the question the previous one raised, in SCQA order. Plan backwards: start from the takeaway and work back to the situation.

**Every chart, every time.** The reader should never have to ask where today sits on the chart, what the trend is, what the regression line is, or what the takeaway is. Mark the event on the chart and put the answer in the title.

**Hand-in.** Your recorded video is the presentation: SCQA order, five charts, everyone speaks. Submit it with the notebook (including a who-did-what table) and the Trello board.

**Which option?** Any of A to D. A, Silicon Valley Bank, has the most support. D, Microsoft, has the least, which leaves the most room for creativity.

## Option A - The Silicon Valley Bank collapse

What happened, in five steps: (1) pre-covid, interest rates were low, but banks heavily invested in bonds; (2) post-covid, inflation arrived and rates rose; (3) bond prices move inversely to rates, so those holdings fell; (4) Tier 1 banks are "too big to fail" so their deposits stayed, while smaller banks saw fearful withdrawals; (5) SVB lacked liquidity to meet withdrawals, was forced to sell bonds it had classified as held-to-maturity, and realised the losses.

* Data code - in the cells below; the note above them lists what they fix.
* Bank data (JPM, C, BAC, SIVB, NYCB, SBNY: date, OHLCV, market cap, 3 January to 31 May 2023) - provided to the class as a file; not included in this repository

Chart ideas from prior cohorts: percentage of uninsured deposits over time; interest rates vs inflation and why SVB was hurt; histogram of daily returns with the collapse annotated (expected or unexpected?); candlesticks around the collapse comparing regional vs Tier 1 banks; treemap sized by market cap and colored by return; spread plot long large banks / short regional banks. A question worth answering in your video: some say the Fed has created "too big to fail" and "too small to save" - do you agree?

## Option B - The 2025 trade war

*(Template and lecture provided to the class; not included here.)*

## Option C - The US economy in sectors and jobs

*(Template and walkthrough provided to the class; not included here.)*

## Option D - Predicting product growth at Microsoft

You are a data analyst in 2019 presenting to management. Using revenue growth by product line ([data table](https://sc.cnbcfm.com/applications/cnbc.com/resources/files/2018/10/24/MS_REVS.PNG), [MSFT 10-K on EDGAR](https://www.sec.gov/Archives/edgar/data/789019/000095017023035122/msft-20230630.htm)), show the product mix (this is the one place a pie chart is invited), visualize growth rates by product, and forecast which products will grow - with a rationale you can defend. You have a crystal ball: you know Azure went on to do very well, and you may use that. Close with a reflection on how much of your accuracy came from the data and how much from hindsight. Structure the video with SCQA. No starter notebook for this option - you build from the data and filings, which is part of the point.

**Submit (all options):** presentation video · artifacts · Trello board

## Option A data: FRED and the bank file

FRED is free and needs no API key. These cells fix four problems found in last section's projects:

1. FRED now names its date column `observation_date`, so code that asks for `DATE` fails.
2. The series come at different frequencies. Joining them and calling `dropna()` keeps only the dates all of them share, which throws most of the data away.
3. The uninsured-deposit benchmark has to be national. The series used before covered banks in U.S. territories only.
4. FRED's corporate bond index starts in late 2023, after the collapse. The bond cell below prices a Treasury from its yield instead.

The bank file also needs care: it runs newest first, stores dates as numbers, and carries over-the-counter quotes for SVB and Signature after both banks failed. The last cell handles all three.

In [ ]:
import pandas as pd

# FRED series for Option A. Each comes at its own frequency.
id2label = {
    'CORESTICKM159SFRBATL': 'Inflation',          # monthly, percent change from a year ago
    'DGS5': 'InterestRate5Y',                      # daily, percent
    'BOGZ1FL763139105Q': 'UninsuredDeposits',      # quarterly, millions of dollars, U.S.-chartered banks
    'DPSACBW027SBOG': 'TotalDeposits',             # weekly, billions of dollars, all commercial banks
}

series = {}
for series_id, label in id2label.items():
    s = pd.read_csv(
        f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}',
        index_col=0, parse_dates=True,   # the first column is the date, now named observation_date
    )[series_id]
    series[label] = pd.to_numeric(s, errors='coerce').dropna()

# Keep the series separate. Joining a daily, a weekly, a monthly and a quarterly
# series and then calling dropna() keeps only the dates all four share.
for label, s in series.items():
    print(f'{label:18s} {s.index.min().date()} to {s.index.max().date()}  {len(s):>6,} rows')

In [ ]:
# Percentage of deposits that are uninsured, nationally
import matplotlib.pyplot as plt

uninsured = series['UninsuredDeposits']                          # millions of dollars, one value per quarter
total = series['TotalDeposits'].resample('QS').last() * 1000     # billions to millions, one value per quarter
share = (100 * uninsured / total).dropna()
share.index = share.index + pd.offsets.QuarterEnd(0)   # FRED labels a quarter by its first day; the level is measured at its end

# The two series cover slightly different sets of banks: read the level as
# approximate and the movement as the story.
recent = share.loc['2015':]
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(recent.index, recent.values)
ax.axvline(pd.Timestamp('2023-03-10'), color='red', linestyle='--')
ax.annotate('SVB fails, 10 Mar 2023', xy=(pd.Timestamp('2023-03-10'), recent.max()), xytext=(8, 0),
            textcoords='offset points', color='red')
ax.set_title('Uninsured deposits as a share of all U.S. bank deposits, quarterly')
ax.set_ylabel('Percent of deposits')
plt.show()

In [ ]:
# What rising rates did to bonds banks already owned.
# FRED's corporate bond index (BAMLCC0A1AAATRIV) only starts in late 2023, after the collapse.
# Price a five-year Treasury from its yield instead: $100 paid in five years is worth 100 / (1 + y)^5 today.
y = series['InterestRate5Y'] / 100
bond_price = 100 / (1 + y) ** 5

window = bond_price.loc['2019':'2023-06']
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(window.index, window.values)
ax.axvline(pd.Timestamp('2023-03-10'), color='red', linestyle='--')
ax.annotate('SVB fails, 10 Mar 2023', xy=(pd.Timestamp('2023-03-10'), window.max()), xytext=(-150, 0),
            textcoords='offset points', color='red')
ax.set_title('Value today of $100 paid by the U.S. Treasury in five years')
ax.set_ylabel('Price, dollars')
plt.show()

**The bank file.** The course file `Banks_SVB_2023.csv` is not redistributed here. To run the next cell, supply your own daily file with a `ticker` column, a `dint` date column stored as YYYYMMDD numbers, open/high/low/close/volume and market capitalization, for JPM, C, BAC, SIVB, NYCB and SBNY.

In [ ]:
# The bank file: JPM, C, BAC, SIVB, NYCB and SBNY, 3 January to 31 May 2023
# Data download removed: the course bank file is not redistributed (see the note above and the README).

banks = pd.read_csv('Banks_SVB_2023.csv', index_col=0)
banks['date'] = pd.to_datetime(banks['dint'].astype(str), format='%Y%m%d')   # dates arrive as numbers, e.g. 20230531
banks = banks.sort_values(['ticker', 'date'])                                # the file runs newest first

# SVB's last exchange trade was 9 March 2023 and Signature's was 10 March. The late-March
# rows are over-the-counter quotes after both banks failed, at under a dollar. Drop them,
# or keep them in a separate chart and say what they are.
last_trade = {'SIVB': '2023-03-09', 'SBNY': '2023-03-10'}
for ticker, last_day in last_trade.items():
    banks = banks[~((banks['ticker'] == ticker) & (banks['date'] > last_day))]

banks.groupby('ticker')['date'].agg(['min', 'max', 'count'])